# Day 2 · Prompts, Memory, and the Python Agents Need

This notebook implements **Today's Build** (Sweet Crumbs stateful chatbot with menu dict, decorator, trimming, and `try/except`) and all five experiments in **Your 10%** from the Day 2 AI Agents Workshop tutorial.

In [ ]:
# Step 0: Install the OpenAI SDK (Groq uses an OpenAI-compatible API format)
!pip install -q openai

In [ ]:
import json
import functools
from openai import OpenAI, RateLimitError, AuthenticationError, APIConnectionError

GROQ_API_KEY = "your_groq_api_key_here"  # Replace with your actual Groq API key

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

MODEL = "openai/gpt-oss-20b"
print("Groq client configured!")

---
## Today's Build: A Sweet Crumbs Chatbot That Remembers
Combines a `MENU` dict, a structured `SYSTEM` prompt (`ROLE`, `FACTS`, `RULES`, `FORMAT`), a `@log_call` decorator on `get_price()`, a `history` list with `MAX_TURNS` trimming, and `try/except` error handling.

In [ ]:
MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}

SYSTEM = f"""ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Menu with prices in rupees: {json.dumps(MENU)}
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time. If it isn't in FACTS, say "I'll check with the owner."
        Remember the customer's name and order during the chat.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"   [log] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper

@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for qty of an item on the menu."""
    return MENU[item.lower().strip()] * qty

history = [{"role": "system", "content": SYSTEM}]
MAX_TURNS = 8

def chat(user_text: str) -> str:
    history.append({"role": "user", "content": user_text})
    recent = [history[0]] + history[1:][-MAX_TURNS:]
    try:
        response = client.chat.completions.create(model=MODEL, messages=recent, temperature=0.3)
        reply = response.choices[0].message.content
    except RateLimitError:
        reply = "We're a little busy. Please try again in a minute."
    except APIConnectionError:
        reply = "I can't reach the server right now. Please try again shortly."
    history.append({"role": "assistant", "content": reply})
    return reply

print("Total for 6 cupcakes:", get_price("cupcake", 6))
print("-" * 50)

# Simulated multi-turn customer conversation
test_conversation = [
    "Hi, my name is Mahith.",
    "Can I get 2 eggless chocolate cakes for tomorrow?",
    "How much will that cost in total?",
    "What is my name and what did I order?",
]
for msg in test_conversation:
    print("You:", msg)
    print("Bot:", chat(msg), "\n")

# Uncomment below to chat interactively:
# while True:
#     text = input("You: ")
#     if text.lower().strip() in ("bye", "exit", "quit"):
#         print("Bot: Thank you for visiting Sweet Crumbs!")
#         break
#     print("Bot:", chat(text))

---
## Task 1: Prompt Parts (`RULES` Impact)
Delete the `RULES` lines and ask *"Can you deliver in 1 hour?"*. Put them back and ask again.

In [ ]:
SYSTEM_NO_RULES = f"""ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Menu with prices in rupees: {json.dumps(MENU)}
        Orders need 24 hours' notice. All cakes have eggless versions.
FORMAT: Under 50 words, friendly."""

question_delivery = "Can you deliver a chocolate cake in 1 hour?"

res_no_rules = client.chat.completions.create(
    model=MODEL,
    temperature=0.3,
    messages=[
        {"role": "system", "content": SYSTEM_NO_RULES},
        {"role": "user", "content": question_delivery},
    ],
)

res_with_rules = client.chat.completions.create(
    model=MODEL,
    temperature=0.3,
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question_delivery},
    ],
)

print("WITHOUT RULES:\n", res_no_rules.choices[0].message.content)
print("-" * 50)
print("WITH RULES:\n", res_with_rules.choices[0].message.content)

---
## Task 2: Few-Shot Classifier
Build a message classifier with three worked examples (`order`, `question`, `complaint`) and test it on five new messages.

In [ ]:
def classify_message(new_message: str) -> str:
    """Classify a customer message into one word: order, question, or complaint."""
    messages = [
        {"role": "system", "content": "Label the customer message with one word: order, question or complaint."},
        # Three worked examples (few-shot)
        {"role": "user", "content": "need 2 plum cakes for friday"},
        {"role": "assistant", "content": "order"},
        {"role": "user", "content": "do you have eggless?"},
        {"role": "assistant", "content": "question"},
        {"role": "user", "content": "the cake was dry yesterday"},
        {"role": "assistant", "content": "complaint"},
        # New message to classify
        {"role": "user", "content": new_message},
    ]
    res = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
    return res.choices[0].message.content.strip().lower()

five_new_messages = [
    "Please book 6 cupcakes for Saturday evening.",
    "What is the price of a chocolate cake?",
    "My cupcake frosting was melted when it arrived.",
    "Can I get a plum cake without egg?",
    "I want 1 chocolate cake and 2 cupcakes tomorrow.",
]

for msg in five_new_messages:
    print(f"{msg:<52} -> {classify_message(msg)}")

---
## Task 3: Structured JSON Output + `try/except`
Extract order fields (`name`, `item`, `qty`, `date`, `eggless`) as JSON and parse it safely with `json.loads()`.

In [ ]:
JSON_SYSTEM = """Extract the order from the customer's message.
Return ONLY valid JSON, no markdown fences or extra text, with these exact keys:
name (string), item (string), qty (integer), date (string), eggless (boolean)."""

raw_order_msg = "hi ravi here, need 2 plum cakes for 24th dec, eggless pls"

res_json = client.chat.completions.create(
    model=MODEL,
    temperature=0.0,
    messages=[
        {"role": "system", "content": JSON_SYSTEM},
        {"role": "user", "content": raw_order_msg},
    ],
)

reply_str = res_json.choices[0].message.content.strip()
print("Raw model string:\n", reply_str)

try:
    # Strip accidental markdown code fences if present
    clean_str = reply_str.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    order_dict = json.loads(clean_str)
    print("\nParsed Python dict:", order_dict)
    unit_price = MENU.get(order_dict["item"].lower().strip(), 0)
    print(f"Calculated Bill: Rs {unit_price * order_dict['qty']}")
except json.JSONDecodeError as e:
    print("Failed to parse JSON from model:", e)

---
##Task 4: Memory Trimming (`MAX_TURNS = 2`)
Set `MAX_TURNS = 2`, tell the bot your name, chat for three turns, then ask your name to observe how sliding-window trimming drops earlier context.

In [ ]:
short_history = [{"role": "system", "content": SYSTEM}]
SHORT_MAX_TURNS = 2  # Keeps only the last 2 messages (1 user + 1 assistant)

def chat_short_memory(user_text: str) -> str:
    short_history.append({"role": "user", "content": user_text})
    recent = [short_history[0]] + short_history[1:][-SHORT_MAX_TURNS:]
    res = client.chat.completions.create(model=MODEL, messages=recent, temperature=0.3)
    reply = res.choices[0].message.content
    short_history.append({"role": "assistant", "content": reply})
    return reply

turns = [
    "Hi, my name is Ravi.",
    "Do you have eggless cakes?",
    "How much is a plum cake?",
    "What is my name?",
]

for t in turns:
    print("You:", t)
    print("Bot:", chat_short_memory(t), "\n")

print("Explanation: With MAX_TURNS = 2, only the system prompt and the most recent 2 messages are sent on each call. Once 3 turns pass, 'My name is Ravi' falls outside the window and is never sent to the stateless model.")

---
##Task 5: Rebuild for Your Own Business
Catalogue as a Python dict, a documented pricing function with `@log_call`, a tailored `SYSTEM` prompt, and a safe stateful `chat()` loop.

In [ ]:
SERVICE_CATALOGUE = {
    "os installation": 300,
    "thermal repasting": 450,
    "ssd upgrade labour": 250,
    "hinge repair": 600,
}

TECH_SYSTEM = f"""ROLE:   You are the front-desk assistant for ByteFix Campus Laptop Care.
FACTS:  Service rates in rupees: {json.dumps(SERVICE_CATALOGUE)}
        Open Mon-Sat, 9:30 AM to 6:30 PM. Hardware parts (screens/batteries) take 48 hours.
        Free 15-minute diagnostics for students with college ID.
RULES:  Never promise same-day screen or battery replacement.
        If a service or issue isn't in FACTS, say "I'll check with our senior technician."
        Remember the customer's name and laptop issue during the conversation.
FORMAT: Under 50 words, friendly and clear."""

@log_call
def estimate_service(service_name: str, laptops: int = 1) -> int:
    """Return the total service charge in rupees for a given service and number of laptops."""
    rate = SERVICE_CATALOGUE.get(service_name.lower().strip(), 0)
    return rate * laptops

tech_history = [{"role": "system", "content": TECH_SYSTEM}]

def tech_chat(user_text: str) -> str:
    tech_history.append({"role": "user", "content": user_text})
    recent = [tech_history[0]] + tech_history[1:][-8:]
    try:
        res = client.chat.completions.create(model=MODEL, messages=recent, temperature=0.2)
        reply = res.choices[0].message.content
    except (RateLimitError, AuthenticationError, APIConnectionError):
        reply = "Our desk system is temporarily busy. Please try again in a moment."
    tech_history.append({"role": "assistant", "content": reply})
    return reply

print("Estimate (2x thermal repasting): Rs", estimate_service("thermal repasting", 2))
print("-" * 50)
for q in [
    "Hi, I'm Konda. My Victus laptop is overheating during gaming.",
    "How much is thermal repasting and can you also replace my cracked screen today?",
    "What was my name and laptop issue again?",
]:
    print("You:", q)
    print("Bot:", tech_chat(q), "\n")